## LESSON 1: ENSEMBLE LEARNING

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
datas={
    "possession":[60, 45, 55, 70, 25, 40,35],
    "shots":[15, 7, 12, 13, 6, 17, 22],
    "shots_on_target":[8, 2, 6, 1, 5, 10, 9],
    "pass_accuracy":[75, 32, 82, 34, 25, 57, 49],
    "win":[1, 0, 1, 0, 0, 1, 1]
}
df=pd.DataFrame(datas)
X=df.drop("win", axis=1)
y=df["win"]
print(X.head())

X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models={
    "logistic_regression: ": LogisticRegression(max_iter=1000),
    "decision_tree_classifier: ":DecisionTreeClassifier(random_state=42),
    "random_forest_classifier :":RandomForestClassifier(n_estimators=100, random_state=42)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    score=model.score(X_test, y_test)
    print(name, score)

ensemble=VotingClassifier(
    estimators=[
        ("dt", DecisionTreeClassifier(random_state=42)),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
        ("lr", LogisticRegression(max_iter=1000))
    ],
    voting="hard"
)
ensemble.fit(X_train, y_train)
y_pred=ensemble.predict(X_test)
print("Ensemble Accuracy: ", ensemble.score(X_test, y_test))
print("Prediction", y_pred)

   possession  shots  shots_on_target  pass_accuracy
0          60     15                8             75
1          45      7                2             32
2          55     12                6             82
3          70     13                1             34
4          25      6                5             25
logistic_regression:  1.0
decision_tree_classifier:  0.5
random_forest_classifier : 1.0
Ensemble Accuracy:  1.0
Prediction [1 0]


# NOTE:
---
~~~ 
Ensemble Learning combines multiple models.
Bagging trains models independently using different samples.
Boosting trains models sequentially to improve predictions.
Stacking combines model predictions using another model.
Random Forest is an example of Bagging.
XGBoost and LightGBM are example of Boosting.
Voting combines predictions using majority voting or probability averaging.
~~~
---

## LESSON 2: Bagging Vs Boosting

In [4]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y=make_classification(
    n_samples=100,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    class_sep=0.8,
    flip_y=0.05,
    random_state=42
)
feature_names=[
    "possession",
    "shots",
    "shots_on_target",
    "pass_accuracy",
    "corners",
    "tackles",
    "fouls",
    "distance_covered"
]
X=pd.DataFrame(X, columns=feature_names)
y=pd.Series(y, name="win")

X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Bagging implementation
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
bagging_model=BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)
bagging_model.fit(X_train, y_train)
bagging_pred= bagging_model.predict(X_test)

# Boosting model
from sklearn.ensemble import AdaBoostClassifier
boosting_model=AdaBoostClassifier(
    estimator=DecisionTreeClassifier(),
    learning_rate=0.5,
    n_estimators=100,
    random_state=42
)
boosting_model.fit(X_train, y_train)
boosting_pred= boosting_model.predict(X_test)

# Comparision
from sklearn.metrics import accuracy_score, f1_score
results=pd.DataFrame({
    "Model":["Bagging", "Boosting(ADABoost)"],
    "Accuracy":[
        accuracy_score(y_test, bagging_pred),
        accuracy_score(y_test, boosting_pred)
    ],
    "F1_score":[
        f1_score(y_test, bagging_pred),
        f1_score(y_test, boosting_pred)
    ]
})

print(results)

                Model  Accuracy  F1_score
0             Bagging      0.85  0.823529
1  Boosting(ADABoost)      0.80  0.777778


# NOTE:
---
~~~ 
Bagging: Independent model training.
Bootstrap: Sampling with replacement.
Variance: Sensitivity to traning data.
Boosting: Sequential model improvement.
AdaBoost: Focuses on misclassified samples.
GradientBoosting: Learns Corrections using Gradients.
RandomForest: Bagging-based ensemble.
XGBoost: Advanced/Optimized Gradient Boosting
LightGBM: Efficient Gradient Boosting Framework.
~~~
---

## LESSON 3: GRADIENT BOOSTING

In [5]:
import pandas as pd
from sklearn.datasets import make_classification
X, y=make_classification(
    n_samples=100,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    class_sep=0.8,
    flip_y=0.05,
    random_state=42
)
feature_names=[
    "possession",
    "shots",
    "shots_on_target",
    "pass_accuracy",
    "corners",
    "tackles",
    "fouls",
    "distance_covered"
]
X=pd.DataFrame(X, columns=feature_names)
y=pd.Series(y, name="win")

# Train/Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# Training Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score

params={
    "n_estimators":[50, 100, 200],
    "learning_rate":[0.01, 0.1, 0.5]
}
g_results=[]
for n in params["n_estimators"]:
    for l in params["learning_rate"]:
        g_model=GradientBoostingClassifier(
            n_estimators=n,
            learning_rate=l,
            random_state=42,
            max_depth=3
        )
        g_model.fit(X_train, y_train)
        g_pred=g_model.predict(X_test)
        print(g_pred)
        g_results.append({
            "Estimator":n,
            "learning Rate": l,
            "Accuracy": accuracy_score(y_test, g_pred),
            "F1 Score": f1_score(y_test, g_pred)
        })
g_results_df=pd.DataFrame(g_results)
print(g_results_df)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
models={
    "DECISION TREE":DecisionTreeClassifier(random_state=42),
    "RANDOM FOREST":RandomForestClassifier(n_estimators=100, random_state=42),
    "ADABOOST":AdaBoostClassifier(n_estimators=100, random_state=42),
    "GRADIENT BOOSTING":GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}
results=[]
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred=model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1_score": f1_score(y_test, y_pred)
    })
results_df=pd.DataFrame(results)
print(results_df.sort_values(by="Accuracy", ascending=False))

[1 0 1 1 1 0 0 0 0 0 1 1 0 0 0 0 1 0 1 1]
[1 0 1 1 1 0 0 0 0 0 0 1 0 0 0 0 1 0 1 1]
[1 0 1 1 1 0 0 0 0 0 0 1 0 0 0 0 1 0 1 1]
[1 0 1 1 0 0 0 0 0 0 1 1 0 0 0 1 1 0 1 1]
[1 0 1 1 1 0 0 0 0 0 0 1 0 0 0 1 1 0 1 1]
[1 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1]
[1 0 1 1 1 0 0 0 0 0 0 1 0 0 0 0 1 0 1 1]
[1 0 1 0 1 0 0 0 0 0 0 1 0 0 0 1 0 0 1 1]
[1 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1]
   Estimator  learning Rate  Accuracy  F1 Score
0         50           0.01      0.75  0.736842
1         50           0.10      0.80  0.777778
2         50           0.50      0.80  0.777778
3        100           0.01      0.85  0.842105
4        100           0.10      0.85  0.842105
5        100           0.50      0.70  0.625000
6        200           0.01      0.80  0.777778
7        200           0.10      0.75  0.705882
8        200           0.50      0.70  0.625000
               Model  Accuracy  F1_score
1      RANDOM FOREST      0.85  0.842105
3  GRADIENT BOOSTING      0.85  0.842105
2           ADABOOS

# NOTE:
---
~~~ 
Gradient Boosting is an ensemble learning technique.
Models are trained sequentially.
Each new model helps reduce the remining errors.
Decision Trees are commonly used as weak learners.
Residual= Actaul Value - Predicted Value (for squared error regression).
Learning rate controls the contribution of each tree.
More tree do not autoatically mean better performance.
XGBoost is an advanced implementation of Gradient Boosting.
~~~
---

## LESSON 4: EXTREME GRADIENT BOOSTING (XGBoosting)

In [6]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
X, y=make_classification(
    n_samples=100,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    class_sep=0.8,
    flip_y=0.05,
    random_state=42
)
feature_names=[
    "possession",
    "shots",
    "shots_on_target",
    "pass_accuracy",
    "corners",
    "tackles",
    "fouls",
    "distance_covered"
]
X=pd.DataFrame(X, columns=feature_names)
y=pd.Series(y, name="win")

X_train, X_test, y_train, t_test= train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42, stratify=y
)

xgb_model= XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    subsample=0.8,
    max_depth=4,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred= xgb_model.predict(X_test)
print(xgb_pred)
print("accuracy_score: ",accuracy_score(y_test, xgb_pred))
print("precsion_score: ",precision_score(y_test, xgb_pred))
print(" recall_score: ",recall_score(y_test, xgb_pred))
print(" f1_score: ",f1_score(y_test, xgb_pred))
print("Classification Report: ",classification_report(y_test, xgb_pred))

models={
    "DECISION TREE":DecisionTreeClassifier(random_state=42),
    "RANDOM FOREST":RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBOOST":XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    "GRADIENT BOOSTING":GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}
results=[]
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred=model.predict(X_test)
    results.append({
        "Models": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })
results_df= pd.DataFrame(results)
print(results_df.sort_values(by="F1 Score", ascending=False))

[1 0 1 1 1 0 0 0 0 0 0 1 0 0 0 1 1 0 1 1]
accuracy_score:  0.85
precsion_score:  0.8888888888888888
 recall_score:  0.8
 f1_score:  0.8421052631578947
Classification Report:                precision    recall  f1-score   support

           0       0.82      0.90      0.86        10
           1       0.89      0.80      0.84        10

    accuracy                           0.85        20
   macro avg       0.85      0.85      0.85        20
weighted avg       0.85      0.85      0.85        20

              Models  Accuracy  Precision  Recall  F1 Score
1      RANDOM FOREST      0.85   0.888889     0.8  0.842105
2            XGBOOST      0.85   0.888889     0.8  0.842105
3  GRADIENT BOOSTING      0.85   0.888889     0.8  0.842105
0      DECISION TREE      0.75   0.777778     0.7  0.736842


## LESSON 5: LIGHTGBM (LIGHT GRADIENT BOOSTING MACHINE )

In [7]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
X, y=make_classification(
    n_samples=100,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=42
)
feature_names=[
    "possession",
    "shots",
    "shots_on_target",
    "pass_accuracy",
    "corners",
    "tackles",
    "fouls",
    "opponent_strength",
    "form",
    "distance_covered"
]
X=pd.DataFrame(X, columns=feature_names)
y=pd.Series(y, name="win")

X_train, X_test, y_train, y_test= train_test_split(
X,
y,
stratify=y,
test_size=0.2,
random_state=42
)
lightGBM_result=[]
for e in [100, 200, 300]:
    for lr in [0.1, 0.05, 0.01]:
        for l in [15, 20, 31]:
            light_model=LGBMClassifier(
                n_estimators=e,
                learning_rate=lr,
                num_leaves=l,
                reg_alpha=0.1,
                reg_lambda=1,
                random_state=42
            )
            light_model.fit(X_train, y_train)
            light_pred= light_model.predict(X_test)
            lightGBM_result.append({
                "Estimators": e,
                "Learning Rate": lr,
                "Number of leaves": l,
                "Accuracy": accuracy_score(y_test, light_pred),
                "Precision": precision_score(y_test, light_pred),
                "Recall": recall_score(y_test, light_pred),
                "F1 Score": f1_score(y_test, light_pred)
            })
lightGBM_result_df=pd.DataFrame(lightGBM_result)
print(lightGBM_result_df)

models={
    "LIGHT GBM":LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=15,reg_alpha=0.1, reg_lambda=1, random_state=42),
    "RANDOM FOREST":RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBOOST":XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    "GRADIENT BOOSTING":GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}
l_results=[]
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred= model.predict(X_test)
    l_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })
l_results_df= pd.DataFrame(l_results)
print(l_results_df.sort_values(by="F1 Score", ascending=False))

    Estimators  Learning Rate  Number of leaves  ...  Precision  Recall  F1 Score
0          100           0.10                15  ...   0.750000     0.9  0.818182
1          100           0.10                20  ...   0.750000     0.9  0.818182
2          100           0.10                31  ...   0.750000     0.9  0.818182
3          100           0.05                15  ...   0.750000     0.9  0.818182
4          100           0.05                20  ...   0.750000     0.9  0.818182
5          100           0.05                31  ...   0.750000     0.9  0.818182
6          100           0.01                15  ...   0.888889     0.8  0.842105
7          100           0.01                20  ...   0.888889     0.8  0.842105
8          100           0.01                31  ...   0.888889     0.8  0.842105
9          200           0.10                15  ...   0.750000     0.9  0.818182
10         200           0.10                20  ...   0.750000     0.9  0.818182
11         200  

# NOTE:
---
~~~ 
LightGBM developed by Microsoft.
Uses Gradient Boosting, leaf-wise tree growth, histogram_based learning.
num_leaves controls tree complexity.
Leaf-wise growth can cause overfitting in small datasets.
Always compare XGBoost and LightGBM experimentally.
~~~
---